# EPH 1T 2026: descarga, filtros y SQLite

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/d-montenegro/eph-agente/blob/main/notebooks/eph_1t2026_sqlite.ipynb)

Primera iteración reproducible para descargar los microdatos oficiales del INDEC, procesarlos sin cargar el archivo completo en memoria y construir una base SQLite. El ZIP descargado se conserva sin modificaciones; los filtros sólo afectan la base derivada.

**Alcance:** exclusivamente EPH continua, 1T 2026. Fuente: Instituto Nacional de Estadística y Censos (INDEC).

## 1. Configuración

Dejá `AGLOMERADOS` vacío y las columnas en `None` para generar la base canónica completa. Para una iteración liviana podés indicar códigos de aglomerado y/o columnas. Las claves de identificación, período y filtrado se agregan siempre.

Una base filtrada sirve para desarrollo y exploración, pero cambia el universo estadístico. No debe presentarse como una estimación nacional ni reemplazar la base completa.

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import os
import sqlite3
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

URLS_FUENTE = [
    "https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/EPH_usu_1_Trim_2026_txt.zip",
    # Respaldo reproducible si el servidor oficial no responde desde Colab.
    "https://raw.githubusercontent.com/d-montenegro/eph-agente/main/datos_indec/EPH_usu_1_Trim_2026_txt.zip",
]
EXPECTED_ZIP_SHA256 = "d9d23945d4b1d3de362e154015fd9abacc76ed641aec6f41a805b809780c0343"
ZIP_PATH = Path("/content/EPH_usu_1_Trim_2026_txt.zip")
DB_PATH = Path("/content/eph_1t2026.sqlite")
MANIFEST_PATH = Path("/content/eph_1t2026.manifest.json")
CHUNK_SIZE = 10_000

# Ejemplo para Gran Córdoba: AGLOMERADOS = {13}
AGLOMERADOS: set[int] = set()

# None conserva todas las columnas. Ejemplo: ["CH04", "CH06", "ESTADO", "PONDERA"]
COLUMNAS_POR_TABLA: dict[str, list[str] | None] = {
    "individual": None,
    "hogar": None,
}


## 2. Descarga auditable

La descarga se hace por bloques, valida que el resultado sea un ZIP y muestra su SHA-256. Si el enlace oficial falla, se usa la copia sin modificar versionada en el repositorio.

In [ ]:
def sha256_archivo(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for bloque in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(bloque)
    return digest.hexdigest()


def descargar(urls: list[str], destino: Path) -> str:
    temporal = destino.with_suffix(destino.suffix + ".part")
    errores: list[str] = []
    for url in urls:
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "eph-agente-colab/1.0"})
            with urllib.request.urlopen(request, timeout=120) as response, temporal.open("wb") as out:
                while bloque := response.read(1024 * 1024):
                    out.write(bloque)
            if not zipfile.is_zipfile(temporal):
                raise ValueError("la respuesta no es un ZIP válido")
            os.replace(temporal, destino)
            return url
        except Exception as exc:
            temporal.unlink(missing_ok=True)
            errores.append(f"{url}: {exc}")
    raise RuntimeError("No se pudo descargar la fuente:\n" + "\n".join(errores))


url_usada = descargar(URLS_FUENTE, ZIP_PATH)
zip_sha256 = sha256_archivo(ZIP_PATH)
if zip_sha256 != EXPECTED_ZIP_SHA256:
    raise RuntimeError(
        "El ZIP descargado cambió respecto de la fuente versionada. "
        "Revisá la actualización metodológica antes de continuar. "
        f"Esperado: {EXPECTED_ZIP_SHA256}; recibido: {zip_sha256}"
    )
print(f"Fuente: {url_usada}")
print(f"Archivo: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.1f} MiB)")
print(f"SHA-256: {zip_sha256}")


## 3. Pipeline incremental

El pipeline hace dos pasadas por cada TXT dentro del ZIP: primero infiere tipos y cuenta filas; luego inserta en SQLite por lotes. El uso de memoria queda acotado aproximadamente por `CHUNK_SIZE`. No se usa `eval` ni SQL aportado por el usuario.

In [ ]:
TABLAS = ("individual", "hogar")
COLUMNAS_OBLIGATORIAS = {"ANO4", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "AGLOMERADO"}


def normalizar_nombre(nombre: str) -> str:
    return nombre.strip().upper().lstrip("\ufeff")


def citar(identificador: str) -> str:
    return '"' + identificador.replace('"', '""') + '"'


def localizar_miembros(zf: zipfile.ZipFile) -> dict[str, str]:
    encontrados: dict[str, str] = {}
    for nombre in zf.namelist():
        base = Path(nombre).name.lower()
        if not base.endswith(".txt"):
            continue
        for tabla in TABLAS:
            if tabla in base:
                if tabla in encontrados:
                    raise ValueError(f"Hay más de un TXT para {tabla}")
                encontrados[tabla] = nombre
    faltantes = set(TABLAS) - encontrados.keys()
    if faltantes:
        raise ValueError(f"Faltan tablas en el ZIP: {sorted(faltantes)}")
    return encontrados


def encabezado(zf: zipfile.ZipFile, miembro: str) -> list[str]:
    with zf.open(miembro) as raw, io.TextIOWrapper(raw, encoding="latin-1") as text:
        frame = pd.read_csv(text, sep=";", nrows=0)
    return [normalizar_nombre(col) for col in frame.columns]


def seleccionar_columnas(tabla: str, disponibles: list[str]) -> list[str]:
    solicitadas = COLUMNAS_POR_TABLA[tabla]
    requeridas = set(COLUMNAS_OBLIGATORIAS)
    if tabla == "individual":
        requeridas.add("COMPONENTE")
    seleccion = set(disponibles if solicitadas is None else map(normalizar_nombre, solicitadas)) | requeridas
    desconocidas = seleccion - set(disponibles)
    if desconocidas:
        raise ValueError(f"Columnas inexistentes en {tabla}: {sorted(desconocidas)}")
    return [col for col in disponibles if col in seleccion]


def iterar_chunks(zf: zipfile.ZipFile, miembro: str, columnas: list[str]):
    seleccion = set(columnas)
    with zf.open(miembro) as raw, io.TextIOWrapper(raw, encoding="latin-1") as text:
        lector = pd.read_csv(
            text,
            sep=";",
            dtype=str,
            keep_default_na=False,
            chunksize=CHUNK_SIZE,
            usecols=lambda col: normalizar_nombre(col) in seleccion,
        )
        for frame in lector:
            frame.columns = [normalizar_nombre(col) for col in frame.columns]
            yield frame[columnas]


def filtrar(frame: pd.DataFrame) -> pd.DataFrame:
    if not AGLOMERADOS:
        return frame
    codigos = {str(codigo) for codigo in AGLOMERADOS}
    return frame[frame["AGLOMERADO"].str.strip().isin(codigos)]


def validar_periodo(frame: pd.DataFrame, tabla: str) -> None:
    if frame.empty:
        return
    periodos = set(zip(frame["ANO4"].str.strip(), frame["TRIMESTRE"].str.strip()))
    if periodos != {("2026", "1")}:
        raise ValueError(f"{tabla} contiene períodos fuera de 2026T1: {sorted(periodos)}")


def perfilar_tabla(zf: zipfile.ZipFile, tabla: str, miembro: str, columnas: list[str]):
    estado = {col: {"numerica": True, "entera": True, "con_valor": False} for col in columnas}
    total = 0
    for frame in iterar_chunks(zf, miembro, columnas):
        frame = filtrar(frame)
        validar_periodo(frame, tabla)
        total += len(frame)
        for col in columnas:
            valores = frame[col].str.strip()
            valores = valores[valores.ne("")]
            if valores.empty:
                continue
            estado[col]["con_valor"] = True
            if col == "CODUSU" or col.endswith("_COD"):
                estado[col]["numerica"] = False
                continue
            numeros = pd.to_numeric(valores.str.replace(",", ".", regex=False), errors="coerce")
            if numeros.isna().any():
                estado[col]["numerica"] = False
                estado[col]["entera"] = False
            elif not (numeros % 1 == 0).all():
                estado[col]["entera"] = False
    if total == 0:
        raise ValueError(f"El filtro dejó {tabla} sin filas")
    tipos = {}
    for col, info in estado.items():
        if info["numerica"] and info["con_valor"]:
            tipos[col] = "INTEGER" if info["entera"] else "REAL"
        else:
            tipos[col] = "TEXT"
    return tipos, total


def convertir(frame: pd.DataFrame, tipos: dict[str, str]) -> pd.DataFrame:
    salida = frame.copy()
    for col, tipo in tipos.items():
        valores = salida[col].str.strip()
        vacios = valores.eq("")
        if tipo == "TEXT":
            salida[col] = valores.where(~vacios, None)
        else:
            numeros = pd.to_numeric(valores.str.replace(",", ".", regex=False), errors="raise")
            numeros[vacios] = pd.NA
            salida[col] = numeros.astype("Int64" if tipo == "INTEGER" else "Float64")
    return salida


def valor_sql(valor: object) -> object:
    if pd.isna(valor):
        return None
    return valor.item() if hasattr(valor, "item") else valor


def construir_sqlite(zip_path: Path, db_path: Path) -> dict:
    temporal = db_path.with_suffix(".tmp.sqlite")
    temporal.unlink(missing_ok=True)
    perfiles: dict[str, dict] = {}
    with zipfile.ZipFile(zip_path) as zf:
        miembros = localizar_miembros(zf)
        for tabla, miembro in miembros.items():
            disponibles = encabezado(zf, miembro)
            columnas = seleccionar_columnas(tabla, disponibles)
            tipos, filas = perfilar_tabla(zf, tabla, miembro, columnas)
            perfiles[tabla] = {"miembro": miembro, "columnas": columnas, "tipos": tipos, "filas": filas}

        con = sqlite3.connect(temporal)
        try:
            con.execute("PRAGMA journal_mode = MEMORY")
            con.execute("PRAGMA synchronous = OFF")
            with con:
                for tabla, perfil in perfiles.items():
                    definiciones = ", ".join(f"{citar(col)} {tipo}" for col, tipo in perfil["tipos"].items())
                    con.execute(f"CREATE TABLE {citar(tabla)} ({definiciones})")
                    columnas_sql = ", ".join(map(citar, perfil["columnas"]))
                    marcas = ", ".join("?" for _ in perfil["columnas"])
                    insert = f"INSERT INTO {citar(tabla)} ({columnas_sql}) VALUES ({marcas})"
                    insertadas = 0
                    for frame in iterar_chunks(zf, perfil["miembro"], perfil["columnas"]):
                        frame = convertir(filtrar(frame), perfil["tipos"])
                        filas = (tuple(valor_sql(v) for v in fila) for fila in frame.itertuples(index=False, name=None))
                        con.executemany(insert, filas)
                        insertadas += len(frame)
                    if insertadas != perfil["filas"]:
                        raise RuntimeError(f"Conteo inconsistente en {tabla}: {insertadas} != {perfil['filas']}")
                    con.execute(f"CREATE INDEX ix_{tabla}_hogar ON {citar(tabla)} (CODUSU, NRO_HOGAR)")
                    con.execute(f"CREATE INDEX ix_{tabla}_aglo ON {citar(tabla)} (AGLOMERADO)")
            con.execute("VACUUM")
        finally:
            con.close()
    os.replace(temporal, db_path)
    return perfiles


## 4. Construcción y manifiesto

El manifiesto registra fuente, huella, filtros, columnas, tipos y conteos. Es el rastro auditable de la base generada.

In [ ]:
perfiles = construir_sqlite(ZIP_PATH, DB_PATH)
modo_completo = not AGLOMERADOS and all(valor is None for valor in COLUMNAS_POR_TABLA.values())
manifest = {
    "dataset": "EPH continua 1T 2026",
    "fuente": url_usada,
    "zip_sha256": zip_sha256,
    "generado_utc": datetime.now(timezone.utc).isoformat(),
    "modo": "completo_canonico" if modo_completo else "filtrado_exploratorio",
    "filtros": {"aglomerados": sorted(AGLOMERADOS)},
    "chunk_size": CHUNK_SIZE,
    "tablas": {
        tabla: {"filas": perfil["filas"], "columnas": perfil["columnas"], "tipos": perfil["tipos"]}
        for tabla, perfil in perfiles.items()
    },
    "advertencias": [
        "Una base filtrada cambia el universo; toda estimación debe documentar filtro, universo y ponderador."
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({tabla: datos["filas"] for tabla, datos in manifest["tablas"].items()}, indent=2))
print(f"SQLite: {DB_PATH} ({DB_PATH.stat().st_size / 1024**2:.1f} MiB)")
print(f"Manifiesto: {MANIFEST_PATH}")


## 5. Verificación rápida

En modo completo se fijan los conteos conocidos del 1T 2026. La consulta de ejemplo es de lectura y está acotada.

In [ ]:
with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as con:
    conteos = {tabla: con.execute(f"SELECT COUNT(*) FROM {citar(tabla)}").fetchone()[0] for tabla in TABLAS}
    muestra = pd.read_sql_query(
        "SELECT ANO4, TRIMESTRE, AGLOMERADO, CODUSU, NRO_HOGAR, COMPONENTE "
        "FROM individual LIMIT 5",
        con,
    )

if modo_completo:
    assert conteos == {"hogar": 15_447, "individual": 43_739}, conteos
print(conteos)
display(muestra)


## 6. Descargar resultados

SQLite es apropiado para esta iteración local y reproducible. Para varios períodos o concurrencia, el siguiente paso es conservar este contrato de ingestión y cambiar el destino a PostgreSQL. Cloudflare D1 debe seguir siendo un adaptador opcional, no parte de la lógica de carga. Antes de sumar otro trimestre hay que revisar cambios en el diseño de registros.

In [ ]:
try:
    from google.colab import files

    files.download(str(DB_PATH))
    files.download(str(MANIFEST_PATH))
except ImportError:
    print("Fuera de Colab; archivos disponibles en:", DB_PATH, MANIFEST_PATH)
